# EN3160 Assignment 01
## Question 1 - Piecewise-linear intensity transformation

A piecewise-linear transformation maps each input intensity to an output intensity by linearly interpolating between specified breakpoints.

In [ ]:
from pathlib import Path

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np


### Function implementation

In [ ]:
def intensity_transform(im, breakpoints):
    """Apply a piecewise-linear transform to a uint8 grayscale or colour image."""
    points = np.asarray(breakpoints, dtype=float)
    if points.ndim != 2 or points.shape[1] != 2 or len(points) < 2:
        raise ValueError('breakpoints must have shape (n, 2), where n >= 2')
    if np.any(points < 0) or np.any(points > 255):
        raise ValueError('breakpoint values must be in [0, 255]')
    if np.any(np.diff(points[:, 0]) <= 0):
        raise ValueError('input intensities must be strictly increasing')

    output = np.interp(im.astype(np.float32), points[:, 0], points[:, 1])
    return np.clip(output, 0, 255).astype(np.uint8)


In [ ]:
# Change these points to experiment with the appearance.
breakpoints = np.array([[0, 0], [50, 38], [100, 145], [150, 192], [255, 255]])

# This works whether VS Code runs the notebook from the project root or a-01.
image_path = next((p for p in (Path('images/emma.jpg'), Path('../images/emma.jpg')) if p.exists()), None)
if image_path is None:
    raise FileNotFoundError('Could not locate images/emma.jpg')

bgr = cv.imread(str(image_path))
original = cv.cvtColor(bgr, cv.COLOR_BGR2RGB)
transformed = intensity_transform(original, breakpoints)

x = np.arange(256)
y = np.interp(x, breakpoints[:, 0], breakpoints[:, 1])
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].plot(x, y, linewidth=2, label='Transformation')
axes[0].scatter(breakpoints[:, 0], breakpoints[:, 1], color='crimson', zorder=3, label='Breakpoints')
axes[0].plot([0, 255], [0, 255], '--', color='gray', label='Identity')
axes[0].set(xlim=(0, 255), ylim=(0, 255), xlabel='Input intensity', ylabel='Output intensity',
            title='Intensity transformation')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].imshow(original); axes[1].set_title('Original image'); axes[1].axis('off')
axes[2].imshow(transformed); axes[2].set_title('Transformed image'); axes[2].axis('off')
fig.tight_layout()
plt.show()


### Observation

The slope is greater than one between 50 and 100, expanding contrast in the darker mid-tones and making facial details clearer. The high-intensity segment has a slope below one, which preserves bright highlights without clipping them.

## Question 2 - Tissue-specific contrast enhancement

The brain proton-density image from Figure 2 is included as `brain_proton_density.bmp`. Two piecewise-linear transforms are used: one expands the brighter tissue range (white matter) and the other expands the mid-tone range (gray matter).

In [ ]:
brain_path = Path('brain_proton_density.bmp')
if not brain_path.exists():
    brain_path = Path('a-01/brain_proton_density.bmp')

brain = cv.imread(str(brain_path), cv.IMREAD_GRAYSCALE)
if brain is None:
    raise FileNotFoundError('Could not locate brain_proton_density.bmp')

# Expand the bright and mid-tone tissue ranges, respectively.
white_matter_points = np.array([[0, 0], [145, 20], [185, 80], [225, 255], [255, 255]])
gray_matter_points = np.array([[0, 0], [70, 0], [105, 45], [165, 255], [255, 255]])

white_enhanced = intensity_transform(brain, white_matter_points)
gray_enhanced = intensity_transform(brain, gray_matter_points)


In [ ]:
x = np.arange(256)
fig, axes = plt.subplots(2, 3, figsize=(13, 8))

for row, (name, points, result) in enumerate([
    ('White matter enhancement', white_matter_points, white_enhanced),
    ('Gray matter enhancement', gray_matter_points, gray_enhanced),
]):
    y = np.interp(x, points[:, 0], points[:, 1])
    axes[row, 0].plot(x, y, linewidth=2, color='tab:blue')
    axes[row, 0].plot([0, 255], [0, 255], '--', color='gray', label='Identity')
    axes[row, 0].scatter(points[:, 0], points[:, 1], color='crimson', zorder=3)
    axes[row, 0].set(title=name, xlim=(0, 255), ylim=(0, 255),
                     xlabel='Input intensity', ylabel='Output intensity')
    axes[row, 0].grid(alpha=0.3)
    axes[row, 0].legend()
    axes[row, 1].imshow(brain, cmap='gray', vmin=0, vmax=255)
    axes[row, 1].set_title('Original proton-density image')
    axes[row, 2].imshow(result, cmap='gray', vmin=0, vmax=255)
    axes[row, 2].set_title(name)
    for ax in axes[row, 1:]:
        ax.axis('off')

fig.tight_layout()
plt.show()


### Observation

The white-matter transform has its steepest segment from 185 to 225, so bright white-matter structures are separated more clearly while darker tissue is compressed. The gray-matter transform instead expands the 105 to 165 mid-tone range, improving the distinction of the cortical gray-matter band and other mid-intensity structures.